In [5]:
import pandas as pd, numpy as np, joblib, torch
from sklearn.metrics import average_precision_score
from src.features import fit_feature_pipeline, transform_features, make_Xy

In [ ]:
# Confirming GPU visibility inside the kernel
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :" , device)

In [ ]:
# building splits and features from raw data
raw = pd.read_parquet("data/train_data_merged.parquet").sort_values("TransactionDT").reset_index(drop=True)
n = len(raw)
tr = raw.iloc[:int(n*0.70)].copy()
va = raw.iloc[int(n*0.70):int(n*0.85)].copy()
te = raw.iloc[int(n*0.85):].copy()

state = fit_feature_pipeline(tr)
X_train , y_train = make_Xy(transform_features(tr, state), state)
X_test , y_test = make_Xy(transform_features(te, state), state)

# Load XGBoost and sanity-check
xgb = joblib.load("models/xgb_baseline.pkl")
test_pr = average_precision_score(y_test, xgb.predict_proba(X_test)[:, 1])
print("XGBoost test PR-AUC:", round(test_pr, 4), "(expect ~0.507)")

# Creating Threat Model and Attack Set

In [12]:
# Exisiting features, grouped by type, to gather info about threat model
feats = state['feature_cols']

# The realistically-manipulable candidate features
candidate_manipulable = [
    "TransactionAmt",       # amount — the most obvious fraudster lever
    "amt_z_for_card",       # (derived from amount — moves with it)
    "hour",                 # timing of the transaction
    "ProductCD_C", "ProductCD_R", "ProductCD_H", "ProductCD_S", "ProductCD_W",  # product type
    "card6_credit", "card6_debit",   # card type used
]

# Which of these are actually in the features set
present = [f for f in candidate_manipulable if f in feats]
print("Candidate manipulable features present:", len(present))
for f in present :
    print("", f)
print("\n Total Features", len(feats))

Candidate manipulable features present: 10
 TransactionAmt
 amt_z_for_card
 hour
 ProductCD_C
 ProductCD_R
 ProductCD_H
 ProductCD_S
 ProductCD_W
 card6_credit
 card6_debit

 Total Features 426


In [13]:
# ---- Threat model ----
# Manipulable: TransactionAmt (amt_z_for_card recomputed as its coupled consequence)
# Everything else: frozen

MANIPULABLE = ["TransactionAmt"]
COUPLED = {"amt_z_for_card" : "TransactionAmt"} # recomputed when amount changes

# Index of the amount feature in the feature matrix
feat_list = state["feature_cols"]
amt_idx = feat_list.index("TransactionAmt")
amtz_idx = feat_list.index("amt_z_for_card")
print(f"TransactionAmt at column {amt_idx}, amt_z_for_card at column {amtz_idx}")

# ---- Building the attack set: the frauds XGBoost currently CATCHES ----
# (you can only 'evade' detection the model currently achieves)
test_probs = xgb.predict_proba(X_test)[:, 1]

# Using a reasonable operating threshold (from Project 1's analysis)
THRESHOLD = 0.30

is_fraud = (y_test.values == 1)
is_caught = (test_probs >= THRESHOLD)
attack_mask = is_fraud & is_caught   # actual frauds flagged by model

X_attack = X_test[attack_mask].copy()
print(f"\nFrauds in test set: {is_fraud.sum():,}")
print(f"Frauds currently CAUGHT (attackable): {attack_mask.sum():,}")
print(f"These are the transactions we'll try to disguise as legitimate.")

TransactionAmt at column 0, amt_z_for_card at column 409

Frauds in test set: 3,083
Frauds currently CAUGHT (attackable): 2,533
These are the transactions we'll try to disguise as legitimate.


In [16]:
# We need card level stats for re-computing amt_z when amount changes
# amt_z = (amount-card_amt_mean)/card_amt_std
# So we need each attack row's card_amt_mean and card_amt_std

amtmean_idx = feat_list.index("card_amt_mean")
amtstd_idx = feat_list.index("card_amt_std")

Xa = X_attack.values.astype(np.float64).copy()   # working array
orig_amounts = Xa[:,amt_idx].copy()
card_means = Xa[:,amtmean_idx]
card_stds = Xa[:,amtstd_idx]

print("Attack Array", Xa.shape)
print("Sample original amounts:", np.round(orig_amounts[:5],2))

# --- Define the pertubation budget ---
# Budget rho = max fraction the amount can change (multiplicative, realistic: a fraudster can scale the charge up or down within reason).
# We will sweep rho in later stage; start by testing a range of candidate amounts per transaction 

def make_adversarial_amounts(orig, rho, n_steps=50):
    """For each original amount, generate candidate amounts within +/- rho fraction."""
    # multiplicative budget: amount can range in [orig*(1-rho), orig*(1+rho)]
    lows  = orig * (1 - rho)
    highs = orig * (1 + rho)
    # candidate grid per transaction: shape (n_txn, n_steps)
    steps = np.linspace(0, 1, n_steps)
    candidates = lows[:, None] + steps[None, :] * (highs - lows)[:, None]
    return candidates   # (n_txn, n_steps)

# Quick test at rho = 0.5 (amount can change up to +/-50%)
cand = make_adversarial_amounts(orig_amounts, rho=0.5, n_steps=50)
print("Candidate grid shape:", cand.shape, "(transactions x amount-options)")
print("For txn 0: original", round(orig_amounts[0],2), "-> range", round(cand[0].min(),2), "to", round(cand[0].max(),2))

Attack Array (2533, 426)
Sample original amounts: [ 46.72 171.   171.   200.   200.  ]
Candidate grid shape: (2533, 50) (transactions x amount-options)
For txn 0: original 46.72 -> range 23.36 to 70.09


In [17]:
def run_amount_attack(Xa, candidates, xgb, threshold, amt_idx, amtz_idx, card_means, card_stds):
    """For each transaction, try all candidate amounts; report if any evades detection."""
    n_txn, n_steps = candidates.shape
    evaded = np.zeros(n_txn, dtype=bool)
    best_adv_amount = Xa[:, amt_idx].copy()  # default: original (if no evasion found)

    # Build one big batch: replicate each row n_steps times with different amounts
    # (vectorized so XGBoost scores everything at once)
    X_big = np.repeat(Xa, n_steps, axis=0)                      # (n_txn*n_steps, 426)
    amt_flat = candidates.reshape(-1)                           # (n_txn*n_steps,)
    X_big[:, amt_idx] = amt_flat
    # Recompute coupled amt_z for the new amounts
    means_rep = np.repeat(card_means, n_steps)
    stds_rep  = np.repeat(card_stds, n_steps)
    X_big[:, amtz_idx] = (amt_flat - means_rep) / stds_rep

    # Score all candidates through XGBoost
    probs = xgb.predict_proba(X_big)[:, 1].reshape(n_txn, n_steps)

    # A candidate 'evades' if its prob drops below threshold
    evades_grid = probs < threshold                            # (n_txn, n_steps)
    evaded = evades_grid.any(axis=1)

    # For evaded txns, record the amount that evaded with least change (optional detail)
    return evaded, probs

evaded, probs_grid = run_amount_attack(
    Xa, cand, xgb, THRESHOLD, amt_idx, amtz_idx, card_means, card_stds
)

n_evaded = evaded.sum()
print(f"Attack budget: ±50% of amount")
print(f"Frauds attacked: {len(evaded):,}")
print(f"Successfully evaded (now classified legit): {n_evaded:,} ({n_evaded/len(evaded)*100:.1f}%)")
print(f"Still caught: {len(evaded)-n_evaded:,}")

Attack budget: ±50% of amount
Frauds attacked: 2,533
Successfully evaded (now classified legit): 106 (4.2%)
Still caught: 2,427


In [18]:
# Sweep attack budget (rho) from 0 upward, measure evasion rate at each
rho_values = [0.0, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0]

results = []
for rho in rho_values:
    cand_r = make_adversarial_amounts(orig_amounts, rho=rho, n_steps=50)
    evaded_r, _ = run_amount_attack(
        Xa, cand_r, xgb, THRESHOLD, amt_idx, amtz_idx, card_means, card_stds
    )
    evasion_rate = evaded_r.mean()
    # Model accuracy on attacked set = fraction still caught
    acc = 1 - evasion_rate
    results.append((rho, evasion_rate, acc))
    print(f"rho = {rho:>4} (±{rho*100:.0f}%) | evaded: {evasion_rate*100:5.1f}% | still caught (Acc): {acc*100:5.1f}%")

print("\n(rho=0 should show 0% evasion — no change allowed = sanity check)")

rho =  0.0 (±0%) | evaded:   0.0% | still caught (Acc): 100.0%
rho =  0.1 (±10%) | evaded:   0.5% | still caught (Acc):  99.5%
rho =  0.2 (±20%) | evaded:   1.1% | still caught (Acc):  98.9%
rho =  0.3 (±30%) | evaded:   1.7% | still caught (Acc):  98.3%
rho =  0.5 (±50%) | evaded:   4.2% | still caught (Acc):  95.8%
rho = 0.75 (±75%) | evaded:   8.3% | still caught (Acc):  91.7%
rho =  1.0 (±100%) | evaded:  12.9% | still caught (Acc):  87.1%
rho =  1.5 (±150%) | evaded:  13.7% | still caught (Acc):  86.3%
rho =  2.0 (±200%) | evaded:  14.1% | still caught (Acc):  85.9%

(rho=0 should show 0% evasion — no change allowed = sanity check)


In [19]:
joblib.dump({
    "attack_mask": attack_mask,
    "results_so_far": results,
}, "models/project2_attack_state.pkl")
print("Saved.")

Saved.
